In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from scipy.spatial.distance import cosine
import joblib
import torch.nn.functional as F

ROBERTA_MODEL = "pysentimiento/robertuito-sentiment-analysis"
EMBED_MODEL = "pysentimiento/robertuito-sentiment-analysis"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREDEF_MESSAGE = "esto es lo peor que me ha pasado, tengo depresion, y no puedo dejar de comer comida poco sana"
RANDOM_SEED = 42


C:\envs\ProyectoMD2_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Carga de modelos
tokenizer_roberta = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model_roberta = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL).to(DEVICE)
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

print("Modelos cargados correctamente")


No sentence-transformers model found with name pysentimiento/robertuito-sentiment-analysis. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at pysentimiento/robertuito-sentiment-analysis and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelos cargados correctamente


In [3]:
def robertuito_probs(texts, batch_size=16):
    """Obtiene las probabilidades [neg, neu, pos] para cada texto."""
    probs = []
    model_roberta.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer_roberta(batch, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
            out = model_roberta(**encoded)
            logits = out.logits
            batch_probs = F.softmax(logits, dim=-1).cpu().numpy()
            probs.append(batch_probs)
    return np.vstack(probs)

predef_emb = SentenceTransformer(EMBED_MODEL, device=DEVICE).encode(PREDEF_MESSAGE, convert_to_numpy=True)

def cosine_distance_batch(texts):
    """Calcula la distancia coseno entre cada texto y el mensaje predefinido."""
    embs = embedder.encode(texts, convert_to_numpy=True)
    dists = [cosine(e, predef_emb) for e in embs]
    return np.array(dists), embs

def build_features(texts):
    """Genera el vector de features concatenando: embeddings + distancia coseno."""
    dists, embs = cosine_distance_batch(texts)
    dists = dists.reshape(-1, 1)
    X = np.hstack([embs, dists])
    return X

def build_features_robertuito_only(texts):
    """Devuelve solo las probabilidades del modelo Robertuito (neg, neu, pos)."""
    probs = robertuito_probs(texts)
    return probs


No sentence-transformers model found with name pysentimiento/robertuito-sentiment-analysis. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at pysentimiento/robertuito-sentiment-analysis and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import pandas as pd

# Ruta del CSV
csv_path = "output/TCA_exploded.csv"

# Leer el archivo
df = pd.read_csv(csv_path, encoding="utf-8")

# Asegurar que no haya valores nulos
df = df.dropna(subset=["text", "label_id"])

# Extraer las columnas que necesitas
texts = df["text"].astype(str).tolist()
labels = df["label_id"].astype(int).to_numpy()

print(f"Datos cargados desde {csv_path}")
print(f"Total de filas: {len(df)}")
print("Ejemplo de texto:", texts[0])
print("Ejemplo de etiqueta:", labels[0])


Datos cargados desde output/TCA_exploded.csv
Total de filas: 10110
Ejemplo de texto: hombre la fuerza de voluntad jodida jajaj
Ejemplo de etiqueta: 1


In [5]:
print("Generando features...")
X = build_features(texts)
y = build_features_robertuito_only(texts)
print("Listo, shape de X:", X.shape)
print("Listo, shape de y:", y.shape)



Generando features...
Listo, shape de X: (10110, 769)
Listo, shape de y: (10110, 3)


In [6]:


if y.ndim == 2 and y.shape[1] > 1:
    y_labels = y.argmax(axis=1)
else:
    y_labels = np.ravel(y)


vc = pd.Series(y_labels).value_counts().sort_index()
print("Distribución original:", vc.to_dict())

# Eliminar clases con menos de 2 muestras
rare = vc[vc < 2].index.tolist()
if rare:
    print("Eliminando clases con <2 muestras:", rare)
    keep_mask = ~pd.Series(y_labels).isin(rare).values
    X = X[keep_mask]
    y_labels = y_labels[keep_mask]
    print("Nuevo shape X:", X.shape, "y:", y_labels.shape)
    vc = pd.Series(y_labels).value_counts().sort_index()
    print("Distribución después:", vc.to_dict())

# split con stratify si es seguro
test_size = 0.2 if len(y_labels) > 10 else 0.4
stratify_opt = y_labels if len(np.unique(y_labels)) > 1 and len(y_labels) > 5 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=test_size, random_state=RANDOM_SEED, stratify=stratify_opt
)

print(f"Tamaño train: {len(X_train)}, test: {len(X_test)}")
print("Distribución train:", pd.Series(y_train).value_counts().to_dict())
print("Distribución test:", pd.Series(y_test).value_counts().to_dict())


Distribución original: {0: 2690, 1: 5655, 2: 1765}
Tamaño train: 8088, test: 2022
Distribución train: {1: 4524, 0: 2152, 2: 1412}
Distribución test: {1: 1131, 0: 538, 2: 353}


In [7]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(random_state=RANDOM_SEED, max_iter=500))
])

param_grid = {
    "mlp__hidden_layer_sizes": [(64,), (128,), (128, 64)],
    "mlp__alpha": [1e-4, 1e-3],
    "mlp__learning_rate_init": [1e-3, 5e-4]
}

print("Entrenando MLP ")

gs = GridSearchCV(pipeline, param_grid, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print("Entrenamiento completado")
print("Mejores hiperparámetros:", gs.best_params_)


Entrenando MLP 
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Entrenamiento completado
Mejores hiperparámetros: {'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (128, 64), 'mlp__learning_rate_init': 0.001}


In [8]:
y_pred = gs.predict(X_test)
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# Guardar el pipeline entrenado
joblib.dump(gs.best_estimator_, "mlp_pipeline_with_robertuito.joblib")
print("Modelo guardado en 'mlp_pipeline_with_robertuito.joblib'")


Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       538
           1       0.95      0.95      0.95      1131
           2       0.93      0.95      0.94       353

    accuracy                           0.95      2022
   macro avg       0.94      0.94      0.94      2022
weighted avg       0.95      0.95      0.95      2022

Matriz de confusión:
[[ 503   34    1]
 [  29 1078   24]
 [   0   19  334]]
Modelo guardado en 'mlp_pipeline_with_robertuito.joblib'


In [9]:
import numpy as np
import pandas as pd


if 'labels' in globals() and np.asarray(labels).ndim == 1:
    y_labels = np.asarray(labels).ravel()
    print("Usando 'labels' original como true_label (1D).")

# Predicciones del modelo (sobre X completo)
y_pred_all = gs.predict(X)
n_pred = len(y_pred_all)
print("n_pred = ", n_pred)

df_resultados = pd.DataFrame({
    "text": texts,
    "true_label": y_labels,
    "pred_label": y_pred_all
})

print("Dataset con etiquetas reales y predichas:")
display(df_resultados.head(20))

print("\nEjemplos mal clasificados:")
display(df_resultados[df_resultados["true_label"] != df_resultados["pred_label"]].head(20))

df_resultados.to_csv("predicciones_con_textos.csv", index=False, encoding="utf-8")
print("\nArchivo guardado: predicciones_con_textos.csv")

Usando 'labels' original como true_label (1D).
n_pred =  10110
Dataset con etiquetas reales y predichas:


,text,true_label,pred_label
0,hombre la fuerza de voluntad jodida jajaj,1,0
1,y tu que tal llevas ma cuarentena por cierti,1,1
2,necesito salirrr a moverme o algo ahg,1,1
3,que mal en verdad,1,0
4,cuando se supone que acaba ?,1,0
5,En españa ni idea ...,1,1
6,aquí cuando sales te pones guantes y mascarill...,1,1
7,bua no había visto todos estos mensajes,1,0
8,yo no me he presentado tampoco todavía ... así...,1,1
9,Yo me llamo mery y soy de españa uwu,1,1



Ejemplos mal clasificados:


,text,true_label,pred_label
0,hombre la fuerza de voluntad jodida jajaj,1,0
3,que mal en verdad,1,0
4,cuando se supone que acaba ?,1,0
7,bua no había visto todos estos mensajes,1,0
11,yo estoy mazo fat por que no consigo nada ... ...,1,0
12,Siempre intento pensar así pero jo pierdo la f...,1,0
14,ya he bajado unos 5 pero igual en 2 meses por ...,1,0
16,"Ya , intento dormir a veces para no comer o be...",1,0
17,tengo bastante ansiedad asu que no me pasa,1,0
19,Manzanas llenan bastante y no engordan nada y ...,1,2



Archivo guardado: predicciones_con_textos.csv


In [10]:
# usar el modelo guardado para nuevas frases
modelo = joblib.load("mlp_pipeline_with_robertuito.joblib")

nuevos_textos = [
    "me encanta este lugar, todo es maravilloso",
    "me quiero tirar por la ventana"
]

X_new = build_features(nuevos_textos)
preds = modelo.predict(X_new)

for t, p in zip(nuevos_textos, preds):
    print(f"Texto: {t} → Predicción: {p}")


Texto: me encanta este lugar, todo es maravilloso → Predicción: 2
Texto: me quiero tirar por la ventana → Predicción: 0
